In [ ]:
print("동기(synchronous) 버전 코드")
import time

def sync_task(name: str, delay: float):
    print(f"{name} 시작")

    for i in range(3):
        time.sleep(delay)  # CPU가 여기서 완전히 멈춤(블로킹)
        print(f"{name} 작업 중... step {i+1}")
    print(f"{name} 끝")

def main():
    # 동기에서는 순차적으로 하나씩 끝날 때까지 기다림
    sync_task("작업1", 0.5)
    sync_task("작업2", 0.8)
    sync_task("작업3", 1.2)

main()

동기(synchronous) 버전 코드
작업1 시작
작업1 작업 중... step 1
작업1 작업 중... step 2
작업1 작업 중... step 3
작업1 끝
작업2 시작
작업2 작업 중... step 1
작업2 작업 중... step 2
작업2 작업 중... step 3
작업2 끝
작업3 시작
작업3 작업 중... step 1
작업3 작업 중... step 2
작업3 작업 중... step 3
작업3 끝


In [ ]:
print("비동기 작업: 각 작업이 여러 번에 걸쳐 로그를 찍음")
import asyncio

async def async_task(name: str, delay: float):
    print(f"{name} 시작")   # 작업 시작

    for i in range(3):
        await asyncio.sleep(delay)   # delay 동안 다른 작업에 양보
        print(f"{name} 작업 중... step {i+1}")

    print(f"{name} 끝")     # 작업 종료

async def main():
    # 서로 다른 속도로 도는 3개의 작업을 동시에 실행
    await asyncio.gather(
        async_task("작업1", 0.5),
        async_task("작업2", 0.8),
        async_task("작업3", 1.2),
    )

# Jupyter/Colab 이면  → 셀 마지막 줄에 이렇게 실행
await main()

비동기 작업: 각 작업이 여러 번에 걸쳐 로그를 찍음
작업1 시작
작업2 시작
작업3 시작
작업1 작업 중... step 1
작업2 작업 중... step 1
작업1 작업 중... step 2
작업3 작업 중... step 1
작업1 작업 중... step 3
작업1 끝
작업2 작업 중... step 2
작업3 작업 중... step 2
작업2 작업 중... step 3
작업2 끝
작업3 작업 중... step 3
작업3 끝


In [13]:
# 웹스크랩핑(3개, 비동기) -> LLM 요약(모델 2개 - 결정적, 창의적, 비동기) -> RAG(Chroma) -> LLM 최종 답변
!pip install -U langchain langchain-core langchain-community
!pip install -U langchain-google-genai google-genai langchain-openai
!pip install -U chromadb python-dotenv
!pip install -U requests beautifulsoup4

In [12]:
import os
import requests
import asyncio
import textwrap
from bs4 import BeautifulSoup
from typing import List, Dict
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb import PersistentClient
from langchain_openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

True

In [14]:
WIKI_URLS = [
    "https://ko.wikipedia.org/wiki/김치찌개",
    "https://ko.wikipedia.org/wiki/인공지능",
    "https://ko.wikipedia.org/wiki/축구"
]

CHROMA_PATH = "./chroma_wiki_rag"
COLLECTION_NAME = "wiki_multi_rag"
EMBED_MODEL = "all-MiniLM-L6-v2"

# LLM : 사실 중심적 모델 (웹문서 요약용)
llm_precise = OpenAI(model="gpt-4o-mini", temperature=0.0)

# LLM2 : 창의적 모델 (웹문서 요약용)
llm_creative = OpenAI(model="gpt-4o-mini", temperature=0.8)

# LLM3 : 최종 답변용 모델
llm_answer = OpenAI(model="gpt-4o-mini", temperature=0.2)

In [ ]:
# 동기 방식
def extract_from_url_syncFunc(url:str) -> List[str]:
  # p tag의 텍스트 추출 함수
  headers = {"User-Agent": "Mozilla/5.0"}
  resp = requests.get(url, headers=headers)

  print(f"[fetch] {url} -> status_code : {resp.status_code}")

  if resp.status_code != 200:
    print(f"요청 실패 : {resp.text[:200]}")
    return

  soup = BeautifulSoup(resp.text, "html.parser")
  paragraphs = soup.find_all("p")

  texts = [
      p.get_text(strip=True)
      for p in paragraphs if p.get_text(strip=True)
  ]
  print(f"found [p] count: {len(texts)}")
  return texts


# 위 동기 함수를 비동기처럼 감싸는 wrapper (스레드풀 사용)
async def extract_from_url_asyncFunc(url:str) -> List[str]:
  loop = asyncio.get_running_loop()    # <--------- 여기서 반복
  texts = await loop.run_in_executor(
      None,     # 기본 ThreadPoolExecutor
      extract_from_url_syncFunc,
      url
  )
  return texts

# LLM 요약 함수
async def summarize_with_llm_asyncFunc(url:str, paragraphs:List[str], max_chars:int=3000) -> Dict[str, str]:
  if not paragraphs:
    return {"precise":"", "creative":""}    # 빈 요약 반환

  raw_text = "\n".join(paragraphs)
  if len(raw_text) > max_chars:    # LLM에게 보낼 메세지가 너무 길면 자르기
    raw_text = raw_text[:max_chars]

  # 요약을 위한 프롬프트 구성
  prompt = f"""
    너는 한국어 위키백과 문서를 잘 요약하는 전문가야.
    아래는 "{url}" 문서의 일부 내용이야.
    핵심개념, 중요한 사실, 특징을 중심으로 10문장 내외의 한국어 요약문을 만들어줘.
    문장은 자연스럽고 중복없이 처리해줘.
    ===원문 일부===
    {raw_text}
  """

  precise_task = llm_precise.ainvoke(prompt)
  creative_task = llm_creative.ainvoke(prompt)

  # 두 요청을 동시에 보내고 둘 다 끝날 때까지 대기
  precise_result, creative_result = await asyncio.gather(precise_task, creative_task)

  return {
      "precise":precise_result.strip(),
      "creative":creative_result.strip(),
  }

# URL 하나에 대한 전체 처리(스크랩핑 + LLM 요약)
async def process_urlFunc(url:str) -> Dict:
  paragraphs = await extract_from_url_asyncFunc(url)
  if not paragraphs:
    return {
        "url":url,
        "paragraphs":[],
        "summary_precise":"",
        "summary_creative":"",
    }
  print(f"[parse] {url} -> 문단 {len(paragraphs)}개 추출")

  summary_dict = await summarize_with_llm_asyncFunc(url, paragraphs)
  print(
      f"{url} 요약 길이 :"
      f"precise = {len(summary_dict['precise'])}"
      f"creative = {len(summary_dict['creative'])}"
  )
  return {
        "url":url,
        "paragraphs":paragraphs,
        "summary_precise":summary_dict["precise"],
        "summary_creative":summary_dict["creative"],
    }


# Embedding + Chroma 초기화
def init_embedding_and_chromaFunc():
  embedder = SentenceTransformer(EMBED_MODEL)
  client = PersistentClient(path=CHROMA_PATH)
  collection = client.get_or_create_collection(name=COLLECTION_NAME)
  return embedder, collection

# LLM 요약들을 ChromaDB에 저장 (LLM 2개 사용) : 각 url당 2개의 문서 (precise, creative)룰 저장
def store_to_chromaFunc(embedder:SentenceTransformer, collection, processed_results:List[Dict]):
  docs = []
  metadatas = []
  ids = []
  for idx, item in enumerate(processed_results):
    url = item["url"]
    sp = item.get("summary_precise", "")
    sc = item.get("summary_creative", "")

    if sp:
      docs.append(sp)
      metadatas.append({
          "url":url,
          "type":"summary_precise",
          })
      ids.append(f"doc_precise_{idx}")

    if sc:
      docs.append(sc)
      metadatas.append({
          "url":url,
          "type":"summary_creative",
          })
      ids.append(f"doc_creative_{idx}")

  if not docs:
    print("DB에 저장할 문서가 없어요")
    return

  print(f"{len(docs)}개 요약문 임베딩 생성중 ...")
  embeddings = embedder.encode(docs, show_progress_bar=True)

  for doc_id, doc, emb, meta in zip(ids, docs, embeddings, metadatas):
    collection.add(
        ids = [doc_id],
        documents = [doc],
        embeddings = [emb.tolist()],
        metadatas = [meta]
    )

  print(f"현재 컬렉션 문서 수 : {collection.count}")



# RAG 검색 + LLM 최종 답변 생성
def answer_with_ragFunc(embedder:SentenceTransformer, collection, query:str, top_k:int=3):
  print(f"\n질문 : {query}")
  q_emb = embedder.encode([query])[0]

  result = collection.query(
      query_embeddings=[q_emb.tolist()],
      n_results=top_k,
      include=["documents", "metadatas", "distances"],
  )

  docs = result["documents"][0]
  metas = result["metadatas"][0]
  dists = result["distances"][0]

  print("\n질문과 유사한 문서 검색 결과 상위 3개")
  context_blocks = []
  for i, (doc, meta, dist) in enumerate(zip(docs, metas, dists), start=1):
    url = meta.get("url", "unknown")
    doc_type = meta.get("type", "unknown")
    doc_type = meta.get("type", "?")
    print("------------")
    print(f"[{i}] URL : {url}")
    print(f"    type:{doc_type}")
    print(f"    distance:{dist:.4f}")
    print("-----요약 내용-----")
    print(textwrap.fill(doc, width=70))
    context_blocks.append(f"[{i}] URL : {url} ({doc_type})\n{doc}")    # LLM 프롬프트용 누적

  context_text = "\n\n".join(context_blocks)

  # 검색된 요약들을 근거로 LLM에게 최종 답변을 요청
  final_prompt = f"""
    너는 RAG 기반의 한국어를 잘 알고 있는 전문가야.
    아래에 있는 요약문들을 근거로 사용자의 질문에 친절하고 정확하게 답변해줘.

    규칙 :
    1. 컨텍스트에 관련된 내용만 간추려서 정리해
    2. 사용자의 질문에 대해 10 문장 이내 크기로 답변해 줘
    3. 답변 마지막 행에 어떤 URL을 참고했는지 간단히 괄호 안에 표시해줘
    4. 모르는 내용은 지어내지 말고 "모르겠네요"라고 말해줘.
    5. '��' 문자는 출력되지 않게 해줘.

    [사용자 질문]
    {query}
    [검색된 Context 요약문]
    {context_text}
  """

  final_response = llm_answer.invoke(final_prompt)
  final_answer = final_response.strip()
  print("최종 답변 :")
  print(textwrap.fill(final_answer, width=70))



# main함수 ( 웹스크랩핑 -> 요약(llm 모델 2개 사용) -> 벡터DB -> LLM 답변 )
async def mainFunc():
  # 1) web scrapping 후 LLM이 요약
  tasks = [process_urlFunc(url) for url in WIKI_URLS]
  processed_results = await asyncio.gather(*tasks)    # 여러 개(3)의 co-routine을 동시에 실행하고 그 결과를 한 번에 리스트로 반환된 결과 기억

  print("\n모든 URL 처리 완료")
  for item in processed_results:
    print(
        f"- {item["url"]}"
        f"문단 {len(item["paragraphs"])}개, "
        f"precise 요약 길이 {len(item['summary_precise'])}"
        f"creative 요약 {len(item['summary_creative'])}"
    )

  # 2) 요약내용 임베딩 후 ChromaDB에 저장
  embedder, collection = init_embedding_and_chromaFunc()
  store_to_chromaFunc(embedder, collection, processed_results)

  # 3) RAG + LLM
  example_queries = [
      "김치찌개가 무엇인지, 특징과 재료를 중심으로 설명해줘",
      "인공지능과 딥러닝의 관계를 쉽게 설명해줘",
      "축구가 어떤 스포츠인지, 간단한 설명, 역사와 특징을 설명해줘"
  ]

  for q in example_queries:
    answer_with_ragFunc(embedder, collection, q, top_k=3)


# Jupyter/Colab 이면  → 셀 마지막 줄에 이렇게 실행
await mainFunc()

# .py 파일에서 실행할 거면:
# if __name__ == "__main__":
#     asyncio.run(main())

[fetch] https://ko.wikipedia.org/wiki/축구 -> status_code : 200
[fetch] https://ko.wikipedia.org/wiki/김치찌개 -> status_code : 200
found [p] count: 9
[parse] https://ko.wikipedia.org/wiki/김치찌개 -> 문단 9개 추출
[fetch] https://ko.wikipedia.org/wiki/인공지능 -> status_code : 200
found [p] count: 91
[parse] https://ko.wikipedia.org/wiki/인공지능 -> 문단 91개 추출
found [p] count: 70
[parse] https://ko.wikipedia.org/wiki/축구 -> 문단 70개 추출
https://ko.wikipedia.org/wiki/축구 요약 길이 :precise = 437creative = 421
https://ko.wikipedia.org/wiki/인공지능 요약 길이 :precise = 463creative = 448
https://ko.wikipedia.org/wiki/김치찌개 요약 길이 :precise = 423creative = 428

모든 URL 처리 완료
- https://ko.wikipedia.org/wiki/김치찌개문단 9개, precise 요약 길이 423creative 요약 428
- https://ko.wikipedia.org/wiki/인공지능문단 91개, precise 요약 길이 463creative 요약 448
- https://ko.wikipedia.org/wiki/축구문단 70개, precise 요약 길이 437creative 요약 421
6개 요약문 임베딩 생성중 ...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

현재 컬렉션 문서 수 : <bound method Collection.count of Collection(name=wiki_multi_rag)>

질문 : 김치찌개가 무엇인지, 특징과 재료를 중심으로 설명해줘

질문과 유사한 문서 검색 결과 상위 3개
------------
[1] URL : https://ko.wikipedia.org/wiki/축구
    type:summary_creative
    distance:0.9695
-----요약 내용-----
===요약문===    축구는 11명의 선수가 한 팀을 이루어 발을 사용해 상대 골대에 공을 넣는 경기로, 세계에서 가장 인기
있는 스포츠이다. 경기는 직사각형의 경기장에서 진행되며, 골키퍼만 팔을 사용할 수 있고, 나머지 선수는 손과 팔을 제외한 신체
부위로만 공을 다�� 수 있다. 경기 종료 시점까지 더 많은 ��점을 올린 팀이 승리하며, 동점일 경우에는 대회 규칙에 따라
승패를 결정하게 된다. 축구의 현대적인 규칙은 1863년 ��글랜드에서 제정된 이후 현재까지 이어져 오며, FIFA가 주관하는
월드컵이 4년마다 개최된다. 축구라는 용어는 ‘공을 발로 ��다’는 의미를 가지며, 일본어에서 유래되었다. 축구의 기원은
기원전 2, 3세기경 중국에서 행해진 축국으로 알려져 있으며, 고대 그리스와 로마에서도 유사한 형태의 경기가 있었다. 현대
축구의 규칙은 19세기
------------
[2] URL : https://ko.wikipedia.org/wiki/축구
    type:summary_precise
    distance:0.9780
-----요약 내용-----
내용이 포함되어 있다. 축구는 현재 세계에서 가장 인기 있는 스포츠로, 11명의 선수가 팀을 이루어 경기를 진행한다. 경기는
직사각형의 경기장에서 이루어지며, 선수들은 주로 발을 사용하여 공을 다��다. 골키퍼만이 손과 팔을 사용할 수 있으며, 나머지
선수들은 신체의 다른 부위를 사용해야 한다. 경기가 끝날 때 더 많은 ��점을 올린 팀이 